# Visual Guardian V2 — Fall Dataset Preprocessing
**Accelerator:** GPU T4 (required — YOLO11n inference)  
**Inputs:**
- `payutch/fall-video-dataset` — original videos
- `fall-verification-output` — manifest CSV from verification notebook

**Output:** `fall_preprocessed/` → save as Kaggle dataset → input for Phase 2 training notebook  
**Expected runtime:** 2–4 hours on T4

In [ ]:
!pip install ultralytics

import re
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

# Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
OUTPUT_DIR   = Path("/kaggle/working/fall_preprocessed")
CROP_SIZE    = 224
PADDING      = 0.20   # 20% bbox padding
SPLIT_RATIO  = {"train": 0.70, "val": 0.15, "test": 0.15}

# Find manifest CSV (output of verification notebook)
MANIFEST_PATH = "/kaggle/input/datasets/mahad811/fall-verification-manifest/fall_verification_manifest.csv"

print(f"Manifest: {MANIFEST_PATH}")

In [ ]:
# ── Load Manifest + Fix Subject Groups ───────────────────────────────────────
df = pd.read_csv(MANIFEST_PATH)

# Filter to valid clips only
df = df[(df["readable"] == True) & (df["frame_count"] >= 32) & (df["duration_s"] >= 1.0)].copy()
df = df.reset_index(drop=True)
print(f"Valid clips loaded: {len(df)}")

def fix_subject_id(row) -> str:
    """
    Montreal multi-camera fix:
      S_F9_cam6.mp4 → 'SCEN_F09'  (all 8 cameras of same scenario stay together)
    All other subject_id values are kept.
    """
    fname = str(row["filename"])
    sid   = row["subject_id"]

    m = re.match(r'S_F(\d+)_cam\d+', fname, re.IGNORECASE)
    if m:
        return f"SCEN_F{int(m.group(1)):02d}"
    if pd.notna(sid):
        return str(sid)
    return f"OTHER_{Path(fname).stem[:8]}"

df["subject_id"] = df.apply(fix_subject_id, axis=1)
print(f"\nSubject groups:")
print(df["subject_id"].value_counts().to_string())
print(f"\nTotal unique groups: {df['subject_id'].nunique()}")

In [ ]:
# ── Subject-Level Split ───────────────────────────────────────────────────────
groups  = sorted(df["subject_id"].unique())
np.random.seed(42)
np.random.shuffle(groups)

n        = len(groups)
n_train  = int(n * SPLIT_RATIO["train"])
n_val    = int(n * SPLIT_RATIO["val"])

train_groups = set(groups[:n_train])
val_groups   = set(groups[n_train:n_train + n_val])
test_groups  = set(groups[n_train + n_val:])

def assign_split(sid):
    if sid in train_groups: return "train"
    if sid in val_groups:   return "val"
    return "test"

df["split"] = df["subject_id"].apply(assign_split)

print(f"Groups  — train: {len(train_groups)}, val: {len(val_groups)}, test: {len(test_groups)}")
print(f"Clips   — train: {(df['split']=='train').sum()}, val: {(df['split']=='val').sum()}, test: {(df['split']=='test').sum()}")
print(f"\nClass balance per split:")
print(df.groupby(["split", "label"]).size().unstack(fill_value=0).to_string())

# Verify no group overlap
for s1, s2 in [("train", "val"), ("train", "test"), ("val", "test")]:
    overlap = set(df[df["split"]==s1]["subject_id"]) & set(df[df["split"]==s2]["subject_id"])
    assert len(overlap) == 0, f"Group overlap between {s1} and {s2}: {overlap}"
print("\n[OK] No group appears in more than one split")

In [ ]:
# ── YOLO11n Preprocessing ─────────────────────────────────────────────────────
print("Loading YOLO11n...")
model = YOLO("yolo11n.pt")

def fix_orientation(frame: np.ndarray) -> np.ndarray:
    """Rotate portrait-orientation frames to landscape."""
    h, w = frame.shape[:2]
    if h > w * 1.5:
        frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
    return frame

def get_crop(frame: np.ndarray) -> tuple[np.ndarray, bool]:
    """YOLO11n person crop. Falls back to resized full frame if no person detected."""
    results = model(frame, verbose=False, classes=[0])
    boxes   = results[0].boxes

    if boxes is not None and len(boxes) > 0:
        best = boxes[boxes.conf.argmax()]
        x1, y1, x2, y2 = best.xyxy[0].cpu().numpy().astype(int)
        h, w    = frame.shape[:2]
        pad_x   = int((x2 - x1) * PADDING)
        pad_y   = int((y2 - y1) * PADDING)
        x1 = max(0, x1 - pad_x); y1 = max(0, y1 - pad_y)
        x2 = min(w, x2 + pad_x); y2 = min(h, y2 + pad_y)
        crop = frame[y1:y2, x1:x2]
        if crop.size > 0:
            return cv2.resize(crop, (CROP_SIZE, CROP_SIZE)), True

    return cv2.resize(frame, (CROP_SIZE, CROP_SIZE)), False


total, processed, no_person = len(df), 0, 0

for _, row in df.iterrows():
    out_dir = OUTPUT_DIR / row["split"] / row["label"] / Path(row["filename"]).stem
    out_dir.mkdir(parents=True, exist_ok=True)

    cap       = cv2.VideoCapture(row["path"])
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = fix_orientation(frame)
        crop, found = get_crop(frame)
        if not found:
            no_person += 1
        cv2.imwrite(str(out_dir / f"frame_{frame_idx:05d}.jpg"), crop)
        frame_idx += 1

    cap.release()
    processed += 1

    if processed % 100 == 0:
        pct = processed * 100 // total
        print(f"  {processed}/{total} clips ({pct}%)  |  no-person frames so far: {no_person}")

print(f"\nDone. Clips: {processed} | No-person frames (fallback used): {no_person}")

In [ ]:
# ── Save Manifests + Output Summary ──────────────────────────────────────────
manifest_out = "/kaggle/working/fall_split_manifest.csv"
df.to_csv(manifest_out, index=False)
print(f"Split manifest saved: {manifest_out}")

print(f"\nOutput structure:")
for split in ["train", "val", "test"]:
    for label in ["fall", "no_fall"]:
        d = OUTPUT_DIR / split / label
        if d.exists():
            n_clips = sum(1 for _ in d.iterdir())
            print(f"  {split}/{label}: {n_clips} clips")

print("\nNext step:")
print("  Save version → Save & Run All → tick 'Save working directory as dataset'")
print("  Name it: fall-preprocessed-v2")
print("  That dataset is the direct input for the Phase 2 training notebook.")

# TRAINING CELLS


In [1]:
!pip install -q tf-models-official "numpy<2"

In [2]:
import os, glob, random, tarfile, urllib.request, logging

# ── Silence ALL verbose TF/ABSL/CUDA messages ─────────────────────────────────
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['ABSL_MIN_LOG_LEVEL']   = '3'
logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

from pathlib import Path
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score
from official.projects.movinet.modeling import movinet as movinet_lib
from official.projects.movinet.modeling import movinet_model

# ── Single GPU setup ───────────────────────────────────────────────────────────
# NOTE: No MirroredStrategy, No XLA jit_compile
# Both clash with tf-models-official's internal tf_keras backend
# Works correctly on both T4 (sm_75) and P100 (sm_60)
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# mixed_float16: full 2x speedup on T4 (Tensor Cores), ~1.2x on P100 (no Tensor Cores)
tf.keras.mixed_precision.set_global_policy('mixed_float16')

print(f'TF version      : {tf.__version__}')
print(f'GPU             : {gpus[0].name if gpus else "CPU (no GPU found!)"}')
print(f'Precision policy: {tf.keras.mixed_precision.global_policy().name}')

TF version      : 2.20.0
GPU             : /physical_device:GPU:0
Precision policy: mixed_float16


In [3]:
# ── Config ────────────────────────────────────────────────────────────────────
CLIP_FRAMES  = 16      # Fall model: 16 frames (seizure uses 32)
STRIDE       = 2       # needs >= 32 raw frames per clip
IMG_SIZE     = 224
SEED         = 42
WEIGHT_DECAY = 1e-4

# Phase A: backbone frozen → low VRAM → can use larger batch
PHASE_A_BATCH_SIZE = 8
# Phase B: backbone unfrozen → must store gradients for 4M params → back to 4
PHASE_B_BATCH_SIZE = 8

PHASE_A_LR     = 1e-3
PHASE_A_EPOCHS = 10   # up from 5 — more head convergence before backbone releases
PHASE_B_LR     = 5e-5  # down from 1e-4 — smaller updates prevent overfitting on unfreeze
PHASE_B_EPOCHS = 20

CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

# DATA_ROOT: combined notebook mode (preprocessing + training in same notebook)
# The fall_preprocessed folder lives in /kaggle/working after the YOLO cropping cells
DATA_ROOT = Path('/kaggle/working/fall_preprocessed')
if not DATA_ROOT.exists():
    raise RuntimeError(f'DATA_ROOT not found: {DATA_ROOT}. Run the preprocessing cells first.')

print(f'DATA_ROOT        : {DATA_ROOT}')
print(f'Phase A batch    : {PHASE_A_BATCH_SIZE}')
print(f'Phase B batch    : {PHASE_B_BATCH_SIZE}  (reduced for full backbone gradient storage)')

DATA_ROOT        : /kaggle/working/fall_preprocessed
Phase A batch    : 8
Phase B batch    : 8  (reduced for full backbone gradient storage)


In [4]:
# ── Collect clip list ─────────────────────────────────────────────────────────
def collect_clips(data_root, split):
    clips = []
    for label_str, label_int in [('fall', 1), ('no_fall', 0)]:
        folder = Path(data_root) / split / label_str
        if not folder.exists():
            print(f'  Warning: {folder} not found')
            continue
        for clip_dir in folder.iterdir():
            if clip_dir.is_dir() and len(list(clip_dir.glob('*.jpg'))) >= CLIP_FRAMES * STRIDE:
                clips.append((str(clip_dir), label_int))
    random.shuffle(clips)
    return clips

train_clips = collect_clips(DATA_ROOT, 'train')
val_clips   = collect_clips(DATA_ROOT, 'val')
test_clips  = collect_clips(DATA_ROOT, 'test')

print(f'Train : {len(train_clips)} clips  (fall={sum(l for _,l in train_clips)}, no_fall={sum(1-l for _,l in train_clips)})')
print(f'Val   : {len(val_clips)} clips')
print(f'Test  : {len(test_clips)} clips')


# ── Augmentations ─────────────────────────────────────────────────────────────
def augment_clip(clip):
    """Spatial augmentations for fall clips. No temporal resampling (unlike seizure)."""
    if tf.random.uniform(()) > 0.5:
        clip = tf.image.flip_left_right(clip)
    clip = tf.image.random_brightness(clip, 0.3)
    clip = tf.image.random_contrast(clip, 0.7, 1.3)
    clip = clip + tf.random.normal(tf.shape(clip), stddev=0.05)
    clip = tf.clip_by_value(clip, 0.0, 1.0)
    return clip

Train : 5073 clips  (fall=2319, no_fall=2754)
Val   : 711 clips
Test  : 950 clips


In [5]:
# ── Data pipeline ─────────────────────────────────────────────────────────────
def load_clip_frames(clip_dir, n_frames, stride, jitter=True):
    frames    = sorted(glob.glob(os.path.join(clip_dir, '*.jpg')))
    n_raw     = len(frames)
    max_start = max(0, n_raw - n_frames * stride)
    start     = random.randint(0, min(2, max_start)) if jitter else 0
    indices   = [min(start + i * stride, n_raw - 1) for i in range(n_frames)]
    clip = []
    for idx in indices:
        raw = tf.io.read_file(frames[idx])
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.cast(img, tf.float32) / 255.0
        clip.append(img)
    return tf.stack(clip)

def make_generator(clip_list, augment=False):
    def gen():
        for clip_dir, label in clip_list:
            clip = load_clip_frames(clip_dir, CLIP_FRAMES, STRIDE, jitter=augment)
            if augment:
                clip = augment_clip(clip)
            yield clip, tf.cast(label, tf.float32)
    return gen

def make_dataset(clip_list, augment=False, batch_size=PHASE_A_BATCH_SIZE):
    ds = tf.data.Dataset.from_generator(
        make_generator(clip_list, augment),
        output_signature=(
            tf.TensorSpec(shape=(CLIP_FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32)
        )
    )
    options = tf.data.Options()
    options.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.DATA
    ds = ds.with_options(options)
    if augment:
        ds = ds.shuffle(512, seed=SEED)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# Phase A datasets — larger batch (backbone frozen = low VRAM)
train_ds_a = make_dataset(train_clips, augment=True,  batch_size=PHASE_A_BATCH_SIZE)
val_ds_a   = make_dataset(val_clips,   augment=False, batch_size=PHASE_A_BATCH_SIZE)

# Phase B datasets — smaller batch (backbone unfrozen = high VRAM for gradients)
train_ds_b = make_dataset(train_clips, augment=True,  batch_size=PHASE_B_BATCH_SIZE)
val_ds_b   = make_dataset(val_clips,   augment=False, batch_size=PHASE_B_BATCH_SIZE)

# Test dataset
test_ds    = make_dataset(test_clips,  augment=False, batch_size=PHASE_B_BATCH_SIZE)

print('Datasets ready.')
print(f'Phase A train: {len(train_clips)} clips in batches of {PHASE_A_BATCH_SIZE}')
print(f'Phase B train: {len(train_clips)} clips in batches of {PHASE_B_BATCH_SIZE}')

I0000 00:00:1774892939.589381     293 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Datasets ready.
Phase A train: 5073 clips in batches of 8
Phase B train: 5073 clips in batches of 8


In [6]:
# ── Download MoViNet-A2 pretrained checkpoint ──────────────────────────────────
CKPT_URL = 'https://storage.googleapis.com/tf_model_garden/vision/movinet/movinet_a2_base.tar.gz'
CKPT_TAR = '/tmp/movinet_a2_base.tar.gz'
CKPT_DIR = '/tmp/movinet_a2_base'

if not os.path.exists(CKPT_DIR):
    print('Downloading MoViNet-A2 Kinetics-600 checkpoint...')
    urllib.request.urlretrieve(CKPT_URL, CKPT_TAR)
    with tarfile.open(CKPT_TAR) as tar:
        tar.extractall('/tmp')
    print('Done.')
else:
    print('Already downloaded.')
print(f'Checkpoint files: {os.listdir(CKPT_DIR)}')

Done.
Checkpoint files: ['checkpoint', 'ckpt-1.index', 'ckpt-1.data-00000-of-00001']


/tmp/ipykernel_293/3878596800.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('/tmp')


In [7]:
# ── Build model ────────────────────────────────────────────────────────────────
# CRITICAL: Use tf_keras (NOT tf.keras) for the outer model class!
# tf-models-official uses tf_keras internally. Mixing tf.keras and tf_keras
# causes the 'model has no trainable weights' bug.
import tf_keras

class BinaryMovinet(tf_keras.Model):
    def __init__(self, backbone, model_name='fall_movinet_a2'):
        super().__init__(name=model_name)
        self.classifier = movinet_model.MovinetClassifier(
            backbone=backbone,
            num_classes=1,
            dropout_rate=0.5    # Fall: slightly less regularization than seizure (larger dataset)
        )

    def call(self, inputs, training=None):
        logits = self.classifier(inputs, training=training)
        return tf.sigmoid(logits)


def build_binary_movinet(trainable_backbone=False):
    # 1. Build Kinetics-600 model to receive pretrained weights
    backbone   = movinet_lib.Movinet(model_id='a2')
    full_model = movinet_model.MovinetClassifier(backbone=backbone, num_classes=600)
    full_model.build([1, CLIP_FRAMES, IMG_SIZE, IMG_SIZE, 3])

    # 2. Load pretrained checkpoint
    ckpt = tf.train.Checkpoint(model=full_model)
    ckpt.restore(tf.train.latest_checkpoint(CKPT_DIR)).expect_partial()
    print('✓ Pretrained Kinetics-600 weights loaded.')

    # 3. Extract backbone and set trainability
    backbone = full_model.backbone
    backbone.trainable = trainable_backbone

    # 4. Build our binary wrapper (tf_keras model tracks tf_keras weights correctly)
    model = BinaryMovinet(backbone)

    # 5. Run dummy forward pass to fully initialize all weights
    _ = model(tf.zeros([1, CLIP_FRAMES, IMG_SIZE, IMG_SIZE, 3]), training=False)

    trainable     = sum(tf.size(v).numpy() for v in model.trainable_weights)
    non_trainable = sum(tf.size(v).numpy() for v in model.non_trainable_weights)
    print(f'Trainable params : {trainable:,}  (head + upper backbone blocks)')
    print(f'Non-trainable    : {non_trainable:,}  (frozen lower backbone)')
    return model


print('Building model...')
model = build_binary_movinet(trainable_backbone=False)

Building model...
✓ Pretrained Kinetics-600 weights loaded.


I0000 00:00:1774892966.940944     293 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Trainable params : 1,314,817  (head + upper backbone blocks)
Non-trainable    : 2,738,754  (frozen lower backbone)


In [8]:
# ── Phase A: Head-only training (backbone frozen, 5 epochs) ───────────────────
# Monitor val_recall (not val_auc) for fall detection — missing a real fall is more
# dangerous than a false alarm. Recall = sensitivity = 'did we catch every fall?'
model.compile(
    optimizer=tf_keras.optimizers.Adam(PHASE_A_LR),
    loss=tf_keras.losses.BinaryCrossentropy(),
    metrics=[
        tf_keras.metrics.BinaryAccuracy(name='acc'),
        tf_keras.metrics.Recall(name='recall'),
        tf_keras.metrics.Precision(name='precision'),
        tf_keras.metrics.AUC(name='auc')
    ]
)

callbacks_a = [
    tf_keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINT_DIR / 'phaseA_epoch{epoch:02d}_loss{val_loss:.3f}.keras'),
        monitor='val_loss', mode='min', save_best_only=True, verbose=1
    ),
    tf_keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=4, restore_best_weights=True, verbose=1
    )
]

print(f'=== Phase A: Head-only ({PHASE_A_EPOCHS} epochs, LR={PHASE_A_LR:.0e}, batch={PHASE_A_BATCH_SIZE}) ===')
history_a = model.fit(
    train_ds_a,
    validation_data=val_ds_a,
    epochs=PHASE_A_EPOCHS,
    callbacks=callbacks_a
)

=== Phase A: Head-only (10 epochs, LR=1e-03, batch=8) ===
Epoch 1/10
    635/Unknown - 328s 426ms/step - loss: 0.3986 - acc: 0.8157 - recall: 0.7788 - precision: 0.8106 - auc: 0.8980
Epoch 1: val_loss improved from inf to 0.48490, saving model to /kaggle/working/checkpoints/phaseA_epoch01_loss0.485.keras
635/635 [==============================] - 406s 549ms/step - loss: 0.3986 - acc: 0.8157 - recall: 0.7788 - precision: 0.8106 - auc: 0.8980 - val_loss: 0.4849 - val_acc: 0.7778 - val_recall: 0.7430 - val_precision: 0.8012 - val_auc: 0.8644
Epoch 2/10
635/635 [==============================] - ETA: 0s - loss: 0.3063 - acc: 0.8667 - recall: 0.8387 - precision: 0.8656 - auc: 0.9406
Epoch 2: val_loss improved from 0.48490 to 0.43530, saving model to /kaggle/working/checkpoints/phaseA_epoch02_loss0.435.keras
635/635 [==============================] - 304s 449ms/step - loss: 0.3063 - acc: 0.8667 - recall: 0.8387 - precision: 0.8656 - auc: 0.9406 - val_loss: 0.4353 - val_acc: 0.7834 - val_reca

In [9]:
# ── Phase B: Full fine-tune (all 4M params unfrozen) ──────────────────────────
# IMPORTANT: batch size MUST drop from 8 to 4 here.
# Unfreezing backbone means TF must store gradients for 4M params simultaneously.
# Running batch=8 with full backbone causes RESOURCE_EXHAUSTED OOM crash.
PHASE_B_BATCH_SIZE = 8

# Unfreeze all layers
for layer in model.layers:
    layer.trainable = True

total_steps = len(train_clips) // PHASE_B_BATCH_SIZE * PHASE_B_EPOCHS
lr_schedule = tf_keras.optimizers.schedules.CosineDecay(PHASE_B_LR, total_steps)

model.compile(
    optimizer=tf_keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=WEIGHT_DECAY),
    loss=tf_keras.losses.BinaryCrossentropy(),   # No label_smoothing for fall (recall is critical)
    metrics=[
        tf_keras.metrics.BinaryAccuracy(name='acc'),
        tf_keras.metrics.Recall(name='recall'),
        tf_keras.metrics.Precision(name='precision'),
        tf_keras.metrics.AUC(name='auc')
    ]
)

callbacks_b = [
    tf_keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINT_DIR / 'phaseB_epoch{epoch:02d}_loss{val_loss:.3f}.keras'),
        monitor='val_loss', mode='min', save_best_only=True, verbose=1
    ),
    tf_keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    )
]

print(f'=== Phase B: Full fine-tune ({PHASE_B_EPOCHS} epochs, batch={PHASE_B_BATCH_SIZE}, AdamW + Cosine LR) ===')
history_b = model.fit(
    train_ds_b,                  # smaller batch dataset
    validation_data=val_ds_b,
    epochs=PHASE_B_EPOCHS,
    callbacks=callbacks_b
)

=== Phase B: Full fine-tune (20 epochs, batch=8, AdamW + Cosine LR) ===
Epoch 1/20
    635/Unknown - 729s 923ms/step - loss: 0.3125 - acc: 0.8711 - recall: 0.8366 - precision: 0.8758 - auc: 0.9377
Epoch 1: val_loss improved from inf to 0.37210, saving model to /kaggle/working/checkpoints/phaseB_epoch01_loss0.372.keras
635/635 [==============================] - 802s 1s/step - loss: 0.3125 - acc: 0.8711 - recall: 0.8366 - precision: 0.8758 - auc: 0.9377 - val_loss: 0.3721 - val_acc: 0.8326 - val_recall: 0.8184 - val_precision: 0.8444 - val_auc: 0.9128
Epoch 2/20
635/635 [==============================] - ETA: 0s - loss: 0.1839 - acc: 0.9233 - recall: 0.8974 - precision: 0.9323 - auc: 0.9788
Epoch 2: val_loss improved from 0.37210 to 0.34608, saving model to /kaggle/working/checkpoints/phaseB_epoch02_loss0.346.keras
635/635 [==============================] - 613s 938ms/step - loss: 0.1839 - acc: 0.9233 - recall: 0.8974 - precision: 0.9323 - auc: 0.9788 - val_loss: 0.3461 - val_acc: 0.8579

In [13]:
# ── Final Test Set Evaluation (Fast GPU Compiled) ─────────────────────────────
print("Running test set inference... (Fast Compiled Mode)\n")
test_probs, test_labels = [], []

for i, (clips_batch, labels_batch) in enumerate(test_ds):
    print(f"\r  Processing batch {i+1} ...", end="")
    probs = model.predict_on_batch(clips_batch).flatten()
    test_probs.extend(probs)
    test_labels.extend(labels_batch.numpy().flatten())

print("\n")
test_probs  = np.array(test_probs)
test_labels = np.array(test_labels).astype(int)

# 1. Diagnostic Sweep: Automatically find the best threshold mathematically
best_f1 = 0
best_thresh = 0.50

print(f'\n--- True Unseen Test Sweep ---')
print(f'{"Threshold":>10}  {"Recall":>8}  {"Precision":>10}  {"F1":>8}')

for thresh in np.arange(0.10, 0.80, 0.05):
    preds     = (test_probs > thresh).astype(int)
    rec       = recall_score(test_labels, preds, zero_division=0)
    prec      = precision_score(test_labels, preds, zero_division=0)
    f_stat    = f1_score(test_labels, preds, zero_division=0)
    
    # Track the absolute best F1 score
    if f_stat > best_f1:
        best_f1 = f_stat
        best_thresh = thresh
        
    print(f'{thresh:>10.2f}  {rec:>8.3f}  {prec:>10.3f}  {f_stat:>8.3f}')

# 2. Automatically apply the best threshold calculate the final results!
FINAL_THRESHOLD = best_thresh
test_preds = (test_probs > FINAL_THRESHOLD).astype(int)

print(f'\n=== Final Test Set Results (Auto-Optimized) ===')
print(f'Best Threshold : {FINAL_THRESHOLD:.2f}')
print(f'AUC            : {roc_auc_score(test_labels, test_probs):.4f}')
print(f'Recall         : {recall_score(test_labels, test_preds):.4f}')
print(f'Precision      : {precision_score(test_labels, test_preds):.4f}')
print(f'Best F1        : {f1_score(test_labels, test_preds):.4f}')


Running test set inference... (Fast Compiled Mode)

  Processing batch 119 ...


--- True Unseen Test Sweep ---
 Threshold    Recall   Precision        F1
      0.10     0.948       0.784     0.858
      0.15     0.934       0.810     0.867
      0.20     0.925       0.815     0.866
      0.25     0.920       0.833     0.874
      0.30     0.915       0.840     0.876
      0.35     0.913       0.852     0.882
      0.40     0.906       0.861     0.883
      0.45     0.896       0.866     0.881
      0.50     0.887       0.874     0.881
      0.55     0.887       0.881     0.884
      0.60     0.870       0.893     0.882
      0.65     0.863       0.908     0.885
      0.70     0.861       0.910     0.885
      0.75     0.849       0.918     0.882

=== Final Test Set Results (Auto-Optimized) ===
Best Threshold : 0.65
AUC            : 0.9606
Recall         : 0.8632
Precision      : 0.9082
Best F1        : 0.8851


In [ ]:
# ── Threshold Sweep on Validation Set ─────────────────────────────────────────
# For fall detection: we want HIGH RECALL (never miss a fall)
# → lower thresholds increase recall at the cost of more false alarms
print('Running threshold sweep on validation set...')
val_probs, val_labels = [], []
for clips_batch, labels_batch in val_ds_b:
    probs = model(clips_batch, training=False).numpy().flatten()
    val_probs.extend(probs)
    val_labels.extend(labels_batch.numpy().flatten())

val_probs  = np.array(val_probs)
val_labels = np.array(val_labels).astype(int)

print(f'Val AUC: {roc_auc_score(val_labels, val_probs):.4f}')
print(f'\n{"Threshold":>10}  {"Recall":>8}  {"Precision":>10}  {"F1":>8}')

sweep_results = []
for thresh in np.arange(0.25, 0.80, 0.05):
    preds     = (val_probs > thresh).astype(int)
    recall    = recall_score(val_labels, preds, zero_division=0)
    precision = precision_score(val_labels, preds, zero_division=0)
    f1        = f1_score(val_labels, preds, zero_division=0)
    print(f'{thresh:>10.2f}  {recall:>8.3f}  {precision:>10.3f}  {f1:>8.3f}')
    sweep_results.append({'threshold': thresh, 'recall': recall, 'precision': precision, 'f1': f1})

sweep_df = pd.DataFrame(sweep_results)
sweep_df.to_csv('/kaggle/working/fall_threshold_sweep.csv', index=False)

# Recommend threshold where recall first hits >= 0.90
high_recall = sweep_df[sweep_df['recall'] >= 0.90]
if not high_recall.empty:
    best = high_recall.loc[high_recall['precision'].idxmax()]
    print(f'\nRecommended threshold: {best["threshold"]:.2f}  '
          f'(Recall={best["recall"]:.3f}, Precision={best["precision"]:.3f}, F1={best["f1"]:.3f})')
else:
    print('\nWARNING: Recall never reached 0.90. Consider running more Phase B epochs.')
print('Saved: fall_threshold_sweep.csv')


In [ ]:
# ── Old Fall Ensemble Inference (Complete Cell) ─────────────────────────────────
!pip install -q timm
import cv2, torch, timm
import numpy as np
from pathlib import Path
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score


In [2]:

# Paths (Adjust fall_ensemble_dir if your uploaded Kaggle dataset name is different)
test_dir = Path('/kaggle/working/fall_preprocessed/test')  
fall_ensemble_dir = Path('/kaggle/input/datasets/mahad811/fall-ensemble/fall_v2_ensemble')

# MUST run on 'cpu' to avoid Kaggle PyTorch/CUDA driver mismatch errors!
device = torch.device('cpu')
print(f"Testing the Old 5-Model Fall PyTorch Ensemble on Device: {device}")

# 1. Load the 5 EfficientNet Models
def load_models(model_dir):
    models = []
    if not model_dir.exists():
        print(f"[!] Warning: Model directory not found: {model_dir}")
        return models
    for fold_path in sorted(model_dir.glob('fold*.pt')):
        model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=2)
        model.load_state_dict(torch.load(fold_path, map_location=device, weights_only=True))
        model.to(device)
        model.eval()
        models.append(model)
    return models

fall_models = load_models(fall_ensemble_dir)

# 2. Math functions from old FallClassifier (Temporal RGB Triplets)
def build_temporal_rgb(frames):
    n_frames = len(frames)
    if n_frames < 3: return None
    t1 = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
    t2 = cv2.cvtColor(frames[n_frames//2], cv2.COLOR_BGR2GRAY)
    t3 = cv2.cvtColor(frames[-1], cv2.COLOR_BGR2GRAY)
    
    temporal_rgb = cv2.resize(np.stack([t1, t2, t3], axis=-1), (224, 224)).astype(np.float32) / 255.0
    
    # ImageNet norm
    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    img_t = torch.tensor((temporal_rgb - mean) / std).permute(2,0,1).unsqueeze(0).float()
    return img_t.to(device)

# 3. Running Inference directly on the raw files
test_probs, test_labels = [], []

print(f"Checking test data at: {test_dir}\n")

# Check all possible folder names for both classes
class_mapping = {
    'normal': 0, 'Normal': 0, 'adl': 0, 'ADL': 0, 'non_fall': 0, 'no_fall': 0,
    'fall': 1, 'Fall': 1, 'falls': 1
}

for clip_dir in test_dir.glob('*/*'):
    if not clip_dir.is_dir(): continue
    
    label_name = clip_dir.parent.name
    if label_name not in class_mapping:
        continue
        
    label_int = class_mapping[label_name]
    frames = sorted(list(clip_dir.glob('*.jpg')))
    
    if len(frames) < 30: 
        continue
        
    loaded_frames = [cv2.imread(str(f)) for f in frames]
    img_t = build_temporal_rgb(loaded_frames)
    if img_t is None: continue
    
    with torch.no_grad():
        probs = []
        for m in fall_models: probs.append(torch.softmax(m(img_t), dim=1))
        if len(probs) == 0: continue
        
        # According to fall_classifier.py, Index 0 is Fall probability
        ens_prob = torch.stack(probs).mean(dim=0)[0, 0].item()
        
    test_probs.append(ens_prob)
    test_labels.append(label_int)


print(f"--- Load Summary ---")
print(f"Total clips evaluated: {len(test_labels)}")
print(f"Normal/No Fall found (label 0): {sum(1 for x in test_labels if x == 0)}")
print(f"Fall found           (label 1): {sum(1 for x in test_labels if x == 1)}")
print(f"--------------------\n")


# 4. Final Comparison
test_probs, test_labels = np.array(test_probs), np.array(test_labels).astype(int)

if sum(1 for x in test_labels if x == 1) > 0 and sum(1 for x in test_labels if x == 0) > 0:
    best_f1, best_thresh = 0, 0
    for t in np.arange(0.1, 0.9, 0.05):
        f1 = f1_score(test_labels, (test_probs > t).astype(int), zero_division=0)
        if f1 > best_f1: best_f1, best_thresh = f1, t

    preds = (test_probs > best_thresh).astype(int)
    print(f'=== OLD 5-MODEL ENSEMBLE RESULTS ON UNSEEN TEST SET ===')
    print(f'Optimal Threshold : {best_thresh:.2f}')
    print(f'AUC               : {roc_auc_score(test_labels, test_probs):.4f}')
    print(f'Recall            : {recall_score(test_labels, preds):.4f}')
    print(f'Precision         : {precision_score(test_labels, preds):.4f}')
    print(f'Best F1           : {f1_score(test_labels, preds):.4f}')
else:
    print("[!] Error: One of the classes is still missing. Please manually check the folder names inside /kaggle/working/fall_preprocessed/test/ !")


Testing the Old 5-Model Fall PyTorch Ensemble on Device: cpu
Checking test data at: /kaggle/working/fall_preprocessed/test

--- Load Summary ---
Total clips evaluated: 950
Normal/No Fall found (label 0): 526
Fall found           (label 1): 424
--------------------

=== OLD 5-MODEL ENSEMBLE RESULTS ON UNSEEN TEST SET ===
Optimal Threshold : 0.55
AUC               : 0.7848
Recall            : 0.7736
Precision         : 0.6457
Best F1           : 0.7039


In [10]:

keras_path = '/kaggle/working/fall_model_best.keras'
print(f"Loading absolute best model from: {keras_path}")

# 1. Load the weights directly from your saved .keras file!
model.load_weights(keras_path)
print("✅ Successfully loaded the absolute best weights!")

# 2. Run Inference using your existing test_ds
print("\nRunning pure MoViNet inference on Unseen Test Set...")
print("Please hold on for ~30 seconds while the GPU processes all clips...")
test_probs, test_labels = [], []

# Grabbing the data from your natively built test set
for i, (clips_batch, labels_batch) in enumerate(test_ds):
    probs = model.predict_on_batch(clips_batch).flatten()
    test_probs.extend(probs)
    test_labels.extend(labels_batch.numpy().flatten())

test_probs = np.array(test_probs)
test_labels = np.array(test_labels).astype(int)

print(f"\n--- Load Summary ---")
print(f"Total clips evaluated: {len(test_labels)}")
print(f"Normal/No Fall found (label 0): {sum(1 for x in test_labels if x == 0)}")
print(f"Fall found           (label 1): {sum(1 for x in test_labels if x == 1)}")
print(f"--------------------\n")


# 3. Final Comparison calculation
if sum(1 for x in test_labels if x == 1) > 0 and sum(1 for x in test_labels if x == 0) > 0:
    best_f1, best_thresh = 0, 0
    for t in np.arange(0.1, 0.9, 0.05):
        f1 = f1_score(test_labels, (test_probs > t).astype(int), zero_division=0)
        if f1 > best_f1: best_f1, best_thresh = f1, t

    preds = (test_probs > best_thresh).astype(int)
    
    print(f'=== NEW MOVI-NET RESULTS ON UNSEEN TEST SET ===')
    print(f'Optimal Threshold : {best_thresh:.2f}')
    print(f'AUC               : {roc_auc_score(test_labels, test_probs):.4f}')
    print(f'Recall            : {recall_score(test_labels, preds):.4f}')
    print(f'Precision         : {precision_score(test_labels, preds):.4f}')
    print(f'Best F1           : {f1_score(test_labels, preds):.4f}')
    
    print("\n💡 Now you have both sets of metrics perfectly verified for your FYP!")
else:
    print("[!] Error: test_ds seems to be empty or missing a class")


Loading absolute best model from: /kaggle/working/fall_model_best.keras
✅ Successfully loaded the absolute best weights!

Running pure MoViNet inference on Unseen Test Set...
Please hold on for ~30 seconds while the GPU processes all clips...

--- Load Summary ---
Total clips evaluated: 950
Normal/No Fall found (label 0): 526
Fall found           (label 1): 424
--------------------

=== NEW MOVI-NET RESULTS ON UNSEEN TEST SET ===
Optimal Threshold : 0.55
AUC               : 0.9701
Recall            : 0.8679
Precision         : 0.9583
Best F1           : 0.9109

💡 Now you have both sets of metrics perfectly verified for your FYP!


In [11]:

# Pointing exactly to your old MoViNet upload!
keras_path = '/kaggle/input/datasets/mahad811/fall-best-old/fall_model_best-old.keras'
print(f"Loading OLD MoViNet model from: {keras_path}")

# 1. Load the weights directly from your saved old .keras file
model.load_weights(keras_path)
print("✅ Successfully loaded the Old MoViNet weights!")

# 2. Run Inference using your existing test_ds
print("\nRunning Old MoViNet inference on Unseen Test Set...")
print("Please hold on for ~30 seconds while the GPU processes all clips...")
test_probs, test_labels = [], []

# Grabbing the data from your natively built test set
for i, (clips_batch, labels_batch) in enumerate(test_ds):
    probs = model.predict_on_batch(clips_batch).flatten()
    test_probs.extend(probs)
    test_labels.extend(labels_batch.numpy().flatten())

test_probs = np.array(test_probs)
test_labels = np.array(test_labels).astype(int)

print(f"\n--- Load Summary ---")
print(f"Total clips evaluated: {len(test_labels)}")
print(f"Normal/No Fall found (label 0): {sum(1 for x in test_labels if x == 0)}")
print(f"Fall found           (label 1): {sum(1 for x in test_labels if x == 1)}")
print(f"--------------------\n")

# 3. Final Comparison calculation
if sum(1 for x in test_labels if x == 1) > 0 and sum(1 for x in test_labels if x == 0) > 0:
    best_f1, best_thresh = 0, 0
    for t in np.arange(0.1, 0.9, 0.05):
        f1 = f1_score(test_labels, (test_probs > t).astype(int), zero_division=0)
        if f1 > best_f1: best_f1, best_thresh = f1, t

    preds = (test_probs > best_thresh).astype(int)
    
    print(f'=== OLD MOVI-NET RESULTS ON UNSEEN TEST SET ===')
    print(f'Optimal Threshold : {best_thresh:.2f}')
    print(f'AUC               : {roc_auc_score(test_labels, test_probs):.4f}')
    print(f'Recall            : {recall_score(test_labels, preds):.4f}')
    print(f'Precision         : {precision_score(test_labels, preds):.4f}')
    print(f'Best F1           : {f1_score(test_labels, preds):.4f}')
    
else:
    print("[!] Error: test_ds seems to be empty or missing a class")


Loading OLD MoViNet model from: /kaggle/input/datasets/mahad811/fall-best-old/fall_model_best-old.keras
✅ Successfully loaded the Old MoViNet weights!

Running Old MoViNet inference on Unseen Test Set...
Please hold on for ~30 seconds while the GPU processes all clips...

--- Load Summary ---
Total clips evaluated: 950
Normal/No Fall found (label 0): 526
Fall found           (label 1): 424
--------------------

=== OLD MOVI-NET RESULTS ON UNSEEN TEST SET ===
Optimal Threshold : 0.55
AUC               : 0.9701
Recall            : 0.8679
Precision         : 0.9583
Best F1           : 0.9109


In [12]:

# Pointing exactly to your old MoViNet upload!
keras_path = '/kaggle/input/datasets/mahad811/fall-current-movienet/fall-curren.keras'
print(f"Loading OLD MoViNet model from: {keras_path}")

# 1. Load the weights directly from your saved old .keras file
model.load_weights(keras_path)
print("✅ Successfully loaded the Old MoViNet weights!")

# 2. Run Inference using your existing test_ds
print("\nRunning Old MoViNet inference on Unseen Test Set...")
print("Please hold on for ~30 seconds while the GPU processes all clips...")
test_probs, test_labels = [], []

# Grabbing the data from your natively built test set
for i, (clips_batch, labels_batch) in enumerate(test_ds):
    probs = model.predict_on_batch(clips_batch).flatten()
    test_probs.extend(probs)
    test_labels.extend(labels_batch.numpy().flatten())

test_probs = np.array(test_probs)
test_labels = np.array(test_labels).astype(int)

print(f"\n--- Load Summary ---")
print(f"Total clips evaluated: {len(test_labels)}")
print(f"Normal/No Fall found (label 0): {sum(1 for x in test_labels if x == 0)}")
print(f"Fall found           (label 1): {sum(1 for x in test_labels if x == 1)}")
print(f"--------------------\n")

# 3. Final Comparison calculation
if sum(1 for x in test_labels if x == 1) > 0 and sum(1 for x in test_labels if x == 0) > 0:
    best_f1, best_thresh = 0, 0
    for t in np.arange(0.1, 0.9, 0.05):
        f1 = f1_score(test_labels, (test_probs > t).astype(int), zero_division=0)
        if f1 > best_f1: best_f1, best_thresh = f1, t

    preds = (test_probs > best_thresh).astype(int)
    
    print(f'=== OLD MOVI-NET RESULTS ON UNSEEN TEST SET ===')
    print(f'Optimal Threshold : {best_thresh:.2f}')
    print(f'AUC               : {roc_auc_score(test_labels, test_probs):.4f}')
    print(f'Recall            : {recall_score(test_labels, preds):.4f}')
    print(f'Precision         : {precision_score(test_labels, preds):.4f}')
    print(f'Best F1           : {f1_score(test_labels, preds):.4f}')
    
else:
    print("[!] Error: test_ds seems to be empty or missing a class")


Loading OLD MoViNet model from: /kaggle/input/datasets/mahad811/fall-current-movienet/fall-curren.keras
✅ Successfully loaded the Old MoViNet weights!

Running Old MoViNet inference on Unseen Test Set...
Please hold on for ~30 seconds while the GPU processes all clips...

--- Load Summary ---
Total clips evaluated: 950
Normal/No Fall found (label 0): 526
Fall found           (label 1): 424
--------------------

=== OLD MOVI-NET RESULTS ON UNSEEN TEST SET ===
Optimal Threshold : 0.65
AUC               : 0.9606
Recall            : 0.8632
Precision         : 0.9082
Best F1           : 0.8851
